# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR<sup>2</sup> Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities such as record sets, fields, and columns are referenced strictly by their `@id` as defined in the Croissant schema.

### Dataset Source
* [Croissant schema JSON-LD](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

The notebook follows a reproducible analysis structure for FAIR datasets.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and objects using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Display dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

List all record sets, fields, and their `@id`s, using the Croissant metadata schema.

In [ ]:
# List record sets and their fields by their `@id`.
print("Available record sets and fields:")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print()

## 3. Data Extraction

Extract data from each available record set (using `@id`), load into DataFrames, and display their columns and a data preview.

In [ ]:
# Choose the main record set(s) by @id
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    # Use the record set @id to reference records
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Columns for RecordSet {rs_id}:")
        print(dataframes[rs_id].columns.tolist())
        print(dataframes[rs_id].head(), "\n")
    else:
        print(f"No records found for RecordSet {rs_id}.")

## 4. Exploratory Data Analysis (EDA)

Perform data filtering, normalization, and grouping on selected fields. All entity references use field `@id` strings.

In [ ]:
# Example: Choose a record set and fields by @id
# List available record set ids to guide field selection
print("Available record set @ids:", record_set_ids)
if record_set_ids:
    main_rs_id = record_set_ids[0]  # Select the primary table
    df = dataframes[main_rs_id]
    # Print available fields (column names)
    print("Columns in DataFrame:", df.columns.tolist())

    # -- Select numeric field (e.g., coverage percentage, molecular weight, etc.), using the field @id --
    # For demo, let's try 'cr:field/coverage', using exact column names if present.
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'if' or pd.api.types.is_numeric_dtype(df[col])]
    print("Numeric fields detected:", candidate_numeric_fields)
    if candidate_numeric_fields:
        numeric_field_id = candidate_numeric_fields[0]  # Pick first available
        threshold = df[numeric_field_id].mean()  # Use mean as an example threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another available field (categorical)
        candidate_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical/group field found.")
    else:
        print("No numeric field detected in this record set.")
else:
    print("No record sets with data available.")

## 5. Visualization

Visualize the distribution of a numeric field and the group means if available.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if record_set_ids:
    df = dataframes[main_rs_id]
    if candidate_numeric_fields:
        # Histogram of numeric field
        field = numeric_field_id
        plt.figure(figsize=(8,4))
        df[field].hist(bins=30)
        plt.xlabel(field)
        plt.ylabel('Frequency')
        plt.title(f'Distribution of {field}')
        plt.show()

        # Bar plot for group means if grouping field available
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            group_means = df.groupby(group_field)[field].mean().sort_values(ascending=False).head(10)
            group_means.plot(kind='bar', figsize=(8,4))
            plt.ylabel(f'Mean {field}')
            plt.title(f'Mean {field} by {group_field} (Top 10)')
            plt.show()

## 6. Conclusion

- This notebook demonstrated how to access metadata, enumerate all record sets and fields by their `@id`, extract and manipulate tabular data, and conduct basic filtering, normalization, grouping, and visualization—**all with field and record set references using unique Croissant schema `@id` values**.
- For reproducibility, always use explicit `@id` identifiers when accessing record sets, fields, or columns using `mlcroissant`.
- The dataset contains protein abundance and related properties extracted from human extracellular vesicles, suitable for downstream bioinformatics and statistical analysis.

For more advanced analysis, consult the [mlcroissant documentation](https://mlcommons.org/croissant/) and the dataset's FAIR record.